# Codification 1 : analyse des nouveaux retards manqués

Pendant de [`analyse_defauts_manques.ipynb`](../analyse_defauts_manques.ipynb) du machine learning de la deuxième itération, appliqué aux **nouveaux retards** : les clients sains en août (-2, -1 ou 0) et en retard en septembre (2 et plus). Ce sont les changements que l'on veut détecter en priorité, et les modèles en retrouvent peu (10 % au mieux dans le 01e). Journal : [`tableau_resultats.md`](tableau_resultats.md).

**Question** : les nouveaux retards qu'**aucun modèle** ne détecte, qui sont-ils ? Deux issues possibles :
1. **ils ressemblent aux clients restés sains** : rien dans les montants ne les distingue ; la banque les a codifiés sur une information que le dataset ne montre pas ;
2. **ils ont un signe distinctif** que les modèles ratent (paiement de septembre faible, mois d'août déjà mal payé, plafond dépassé…) : il y aurait de quoi aider les modèles.

**Base** : le scénario retenu, [`01e_optimisation_tout_regroupe.ipynb`](01e_optimisation_tout_regroupe.ipynb) (mêmes données, même découpage, mêmes 5 plis, montants et plafond au logarithme, codifications regroupées), ses cinq modèles avec leurs **meilleurs réglages** (journal, passage 9), sans nouvelle optimisation. `dpnm` n'est utilisé à aucun moment. Le test reste de côté : l'analyse porte sur les prédictions hors pli du train.

## 1. Données

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")
df = df.drop(columns="dpnm")   # jamais utilisé dans ce notebook
PAY = [f"PAY_{n}" for n in range(1, 7)]
print(f"Clients du niveau 0 : {len(df)} ; colonnes : {df.shape[1]} (dpnm retirée) ; PAY_1 = 1 : {int((df['PAY_1'] == 1).sum())}")

# Codifications 1 d'avril à août : leurs clients ont tous PAY_1 = 1, ils restent donc hors du train et du test
anciens_1 = (df[PAY[1:]] == 1).any(axis=1)
assert (df.loc[anciens_1, "PAY_1"] == 1).all()
print(f"Clients avec une codification 1 d'avril à août : {int(anciens_1.sum())}, tous codifiés 1 aussi en septembre")

Clients du niveau 0 : 30000 ; colonnes : 24 (dpnm retirée) ; PAY_1 = 1 : 3688
Clients avec une codification 1 d'avril à août : 28, tous codifiés 1 aussi en septembre


## 2. Le découpage (identique au 01e)

Les clients dont `PAY_1` n'est pas un 1, coupés en 80/20 (train / test, `stratify` sur `PAY_1`, `random_state=42`), comme dans les notebooks 01 à 01g ; puis les codifications regroupées (2 et plus → 2), comme dans le 01e.

In [2]:
from sklearn.model_selection import train_test_split

# Variables : colonnes du dataset d'origine (niveau 0), sauf ID (identifiant), PAY_1 (la cible), dpnm et les données personnelles ; aucune colonne ajoutée
PERSONNELLES = ["SEX", "EDUCATION", "MARRIAGE", "AGE"]   # retirées : non pertinentes pour une codification de paiement
NUMERIQUES = ["LIMIT_BAL"] + [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)]
CATEGORIELLES = PAY[1:]
VARIABLES = CATEGORIELLES + NUMERIQUES
assert sorted(VARIABLES + ["ID", "PAY_1"] + PERSONNELLES) == sorted(df.columns)
mask_1 = df["PAY_1"] == 1
connus = df[~mask_1]
a_trancher = df[mask_1]

X_train, X_test, y_train, y_test = train_test_split(
    connus[VARIABLES], connus["PAY_1"], test_size=0.2, stratify=connus["PAY_1"], random_state=42)
CLASSES = sorted(connus["PAY_1"].unique())   # toutes les codifications de PAY_1, sauf 1

repartition = pd.DataFrame({
    "Train": y_train.value_counts(),
    "Test": y_test.value_counts(),
}).reindex(CLASSES).fillna(0).astype(int)
repartition.loc["Total"] = repartition.sum()
print(f"PAY_1 connus : {len(connus)} (train {len(X_train)}, test {len(X_test)}) ; PAY_1 = 1 à trancher : {len(a_trancher)}")
print(f"Variables ({len(VARIABLES)}) : {VARIABLES}")
display(repartition)

PAY_1 connus : 26312 (train 21049, test 5263) ; PAY_1 = 1 à trancher : 3688
Variables (18) : ['PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'LIMIT_BAL', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


,Train,Test
PAY_1,,
-2,2207,552
-1,4549,1137
0,11789,2948
2,2133,534
3,258,64
4,61,15
5,21,5
6,9,2
7,7,2


In [3]:
def regrouper(codifications):
    # 2 et plus -> 2 (retard) ; -2, -1, 0 inchangés
    return codifications.where(codifications < 2, 2)

y_train, y_test = regrouper(y_train), regrouper(y_test)
for col in PAY[1:]:
    X_train[col], X_test[col] = regrouper(X_train[col]), regrouper(X_test[col])
CLASSES = sorted(y_train.unique())
print(f"Cible PAY_1 regroupée : classes {CLASSES}")
print(y_train.value_counts().reindex(CLASSES).rename("Train").to_frame().join(y_test.value_counts().rename("Test")))
print("Variables PAY_2 à PAY_6 regroupées : valeurs", sorted(pd.unique(X_train[PAY[1:]].values.ravel())))

Cible PAY_1 regroupée : classes [np.int64(-2), np.int64(-1), np.int64(0), np.int64(2)]
       Train  Test
PAY_1             
-2      2207   552
-1      4549  1137
 0     11789  2948
 2      2504   626
Variables PAY_2 à PAY_6 regroupées : valeurs [np.int64(-2), np.int64(-1), np.int64(0), np.int64(2)]


## 3. Les modèles du 01e, avec leurs meilleurs réglages

In [4]:
# ==============================================================================
# Score optimisé (choix validé) : "familles équilibrées" ; autre possibilité : "bonnes prédictions"
# ==============================================================================
SCORE = "familles équilibrées"
SURAPPRENTISSAGE_MAX = 7.0   # écart relatif train - validation, en %
VARIANCE_MAX = 2.0           # écart-type des 5 plis, en points

import time
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, make_scorer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

FAMILLES = [-2, -1, 0, 2]
NOMS_FAMILLES = {-2: "-2", -1: "-1", 0: "0", 2: "2 et plus"}

def famille_pay(codifications):
    # Codification -> famille : -2, -1, 0 gardés, 2 et plus réunis en 2
    v = np.asarray(codifications).ravel().astype(int)
    return np.where(v >= 2, 2, v)

def score_familles(vrai, predit):
    v, p = famille_pay(vrai), famille_pay(predit)
    return balanced_accuracy_score(v, p) if SCORE == "familles équilibrées" else (v == p).mean()

assert SCORE in ("familles équilibrées", "bonnes prédictions")
scoreur = make_scorer(score_familles)

def bonnes_predictions(vrai, predit):
    # Pourcentage de bonnes prédictions (bonne famille), en tout et dans chaque famille
    v, p = famille_pay(vrai), famille_pay(predit)
    resultat = {"global": (v == p).mean() * 100}
    for f in FAMILLES:
        resultat[f] = (p[v == f] == f).mean() * 100
    return resultat

# Logarithme signé des montants dus, payés et du plafond, comme dans 01b
MONTANTS_DUS_PAYES = [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)] + ["LIMIT_BAL"]

def log_montants(X):
    X = X.copy()
    X[MONTANTS_DUS_PAYES] = np.sign(X[MONTANTS_DUS_PAYES]) * np.log1p(np.abs(X[MONTANTS_DUS_PAYES]))
    return X

def etape_log():
    return FunctionTransformer(log_montants)

def preprocesseur_lineaire():
    # Codifications : une colonne par valeur ; nombres (déjà au logarithme) : mise à l'échelle standard
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
        ("nombres", StandardScaler(), NUMERIQUES),
    ])

def preprocesseur_arbres():
    # Catégories : une colonne par valeur ; nombres tels quels (déjà au logarithme)
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
    ], remainder="passthrough")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Score optimisé : {SCORE} ; surapprentissage max {SURAPPRENTISSAGE_MAX} % relatif ; alerte si écart-type des plis > {VARIANCE_MAX} points")

Score optimisé : familles équilibrées ; surapprentissage max 7.0 % relatif ; alerte si écart-type des plis > 2.0 points


In [5]:
ESPACES = {
    "Régression logistique": {
        "modele": make_pipeline(etape_log(), preprocesseur_lineaire(), LogisticRegression(max_iter=5000)),
        "n_jobs": -1,
        "echelles": {"logisticregression__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300]},
        "depart": {"logisticregression__C": 1},
        "choix": {"logisticregression__class_weight": [None, "balanced"]},
    },
    "Arbre de décision": {
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(), DecisionTreeClassifier(random_state=42)),
        "n_jobs": -1,
        "echelles": {"decisiontreeclassifier__max_depth": [3, 4, 5, 6, 8, 10, 12, 15, 20, None],
                     "decisiontreeclassifier__min_samples_leaf": [1, 2, 5, 10, 20, 50, 100, 200]},
        "depart": {"decisiontreeclassifier__max_depth": 5, "decisiontreeclassifier__min_samples_leaf": 1},
        "choix": {"decisiontreeclassifier__class_weight": [None, "balanced"],
                  "decisiontreeclassifier__criterion": ["gini", "entropy"]},
    },
    "RandomForest": {
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(), RandomForestClassifier(random_state=42, n_jobs=-1)),
        "n_jobs": -1,
        "echelles": {"randomforestclassifier__n_estimators": [100, 200, 300, 500, 1000],
                     "randomforestclassifier__max_depth": [6, 8, 10, 12, 15, 20, 25, None],
                     "randomforestclassifier__min_samples_leaf": [1, 2, 3, 5, 10, 20, 50]},
        "depart": {"randomforestclassifier__n_estimators": 100, "randomforestclassifier__max_depth": None, "randomforestclassifier__min_samples_leaf": 1},
        "choix": {"randomforestclassifier__max_features": ["sqrt", 0.5],
                  "randomforestclassifier__class_weight": [None, "balanced_subsample"]},
    },
    "MLP": {
        "modele": make_pipeline(etape_log(), preprocesseur_lineaire(), MLPClassifier(max_iter=500, early_stopping=True, random_state=42)),
        "n_jobs": -1,
        "echelles": {"mlpclassifier__hidden_layer_sizes": [(32,), (64,), (100,), (128, 64), (256, 128), (256, 128, 64)],
                     "mlpclassifier__alpha": [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]},
        "depart": {"mlpclassifier__hidden_layer_sizes": (100,), "mlpclassifier__alpha": 1e-4},
        "choix": {},
    },
    "CatBoost": {
        # Réglé comme dans le machine learning de la deuxième itération : même pipeline que les arbres (codifications en une colonne par valeur),
        # tous les cœurs, pas de fichiers écrits ; poids des classes toujours actifs (CatBoost refuse auto_class_weights=None passé explicitement)
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(),
                                CatBoostClassifier(loss_function="MultiClass", iterations=1000, random_seed=42, verbose=0,
                                                   thread_count=-1, allow_writing_files=False)),
        "n_jobs": -1,
        "echelles": {"catboostclassifier__depth": [2, 3, 4, 5, 6, 7, 8, 10],
                     "catboostclassifier__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.1, 0.2],
                     "catboostclassifier__l2_leaf_reg": [1, 3, 5, 10, 15, 30, 60]},
        "depart": {"catboostclassifier__depth": 4, "catboostclassifier__learning_rate": 0.05, "catboostclassifier__l2_leaf_reg": 10},
        "choix": {"catboostclassifier__auto_class_weights": ["Balanced", "SqrtBalanced"]},
    },
}
for nom, e in ESPACES.items():
    for p, v in e["depart"].items():
        assert v in e["echelles"][p], (nom, p)
print("Modèles :", list(ESPACES))

Modèles : ['Régression logistique', 'Arbre de décision', 'RandomForest', 'MLP', 'CatBoost']


In [6]:
# Meilleurs réglages du 01e (journal, passage 9)
MEILLEURS_01E = {
    "Régression logistique": {"logisticregression__C": 10, "logisticregression__class_weight": "balanced"},
    "Arbre de décision": {"decisiontreeclassifier__class_weight": "balanced", "decisiontreeclassifier__criterion": "gini",
                          "decisiontreeclassifier__max_depth": 8, "decisiontreeclassifier__min_samples_leaf": 2},
    "RandomForest": {"randomforestclassifier__class_weight": "balanced_subsample", "randomforestclassifier__max_depth": 25,
                     "randomforestclassifier__max_features": 0.5, "randomforestclassifier__min_samples_leaf": 5,
                     "randomforestclassifier__n_estimators": 500},
    "MLP": {"mlpclassifier__alpha": 1e-4, "mlpclassifier__hidden_layer_sizes": (128, 64)},
    "CatBoost": {"catboostclassifier__auto_class_weights": "Balanced", "catboostclassifier__depth": 5,
                 "catboostclassifier__l2_leaf_reg": 5, "catboostclassifier__learning_rate": 0.03},
}

predictions_hors_pli = {}
for nom, params in MEILLEURS_01E.items():
    modele = ESPACES[nom]["modele"].set_params(**params)
    predictions_hors_pli[nom] = np.asarray(cross_val_predict(modele, X_train, y_train, cv=cv, n_jobs=ESPACES[nom]["n_jobs"])).ravel().astype(int)
    print(f"{nom} : fait")

Régression logistique : fait


Arbre de décision : fait


RandomForest : fait


MLP : fait


CatBoost : fait


**Comment lire** : chaque modèle prédit chaque client du train une fois, par le pli qui ne l'a pas appris (mêmes 5 plis que le 01e). Les résultats doivent retrouver ceux du 01e (passage 9 du journal), à de petits écarts près pour les modèles dont l'apprentissage a une part de hasard.

## 3 bis. L'importance des variables

**Importance par permutation** : pour chaque variable, ses valeurs sont mélangées entre les clients de la validation, et l'on mesure **la baisse du score** (moyenne des familles, en points). Une variable dont le mélange ne fait rien perdre n'est pas utilisée par le modèle. Le calcul se fait pli par pli, sur les clients que le modèle n'a pas appris (mêmes 5 plis, 5 mélanges par variable), puis la moyenne des plis est affichée. Les variables d'origine sont mélangées entières (une codification avec toutes ses valeurs).

Le second tableau regroupe les variables **par mois** : septembre (M : `BILL_AMT1`, `PAY_AMT1`), août (M-1 : `PAY_2`, `BILL_AMT2`, `PAY_AMT2`), juillet (M-2), juin (M-3), mai (M-4), avril (M-5), et le plafond. Il répond à la question : les mois anciens servent-ils à quelque chose ?

In [7]:
from sklearn.base import clone
from sklearn.inspection import permutation_importance

REGLAGES = MEILLEURS_01E
NOMS_MOIS = {1: "septembre (M)", 2: "août (M-1)", 3: "juillet (M-2)", 4: "juin (M-3)", 5: "mai (M-4)", 6: "avril (M-5)"}

def mois_de(variable):
    return "plafond" if variable == "LIMIT_BAL" else NOMS_MOIS[int(variable[-1])]

importances = {}
for nom, params in REGLAGES.items():
    par_pli = []
    for apprentissage, validation in cv.split(X_train, y_train):
        m = clone(ESPACES[nom]["modele"]).set_params(**params).fit(X_train.iloc[apprentissage], y_train.iloc[apprentissage])
        r = permutation_importance(m, X_train.iloc[validation], y_train.iloc[validation], scoring=scoreur,
                                   n_repeats=5, random_state=42, n_jobs=-1)
        par_pli.append(r.importances_mean * 100)
    importances[nom] = pd.Series(np.mean(par_pli, axis=0), index=X_train.columns)
    print(f"{nom} : fait")

table = pd.DataFrame(importances)
table["Moyenne des modèles"] = table.mean(axis=1)
display(Markdown("**Par variable** (baisse du score en points, de la plus importante à la moins importante en moyenne)"))
display(table.sort_values("Moyenne des modèles", ascending=False).round(2))
par_mois = table.groupby(table.index.map(mois_de)).sum()
ordre = [NOMS_MOIS[n] for n in range(1, 7)] + ["plafond"]
display(Markdown("**Par mois** (somme des variables du mois)"))
display(par_mois.reindex(ordre).round(2))

Régression logistique : fait


Arbre de décision : fait


RandomForest : fait


MLP : fait


CatBoost : fait


**Par variable** (baisse du score en points, de la plus importante à la moins importante en moyenne)

,Régression logistique,Arbre de décision,RandomForest,MLP,CatBoost,Moyenne des modèles
PAY_2,49.75,52.22,53.92,41.07,46.17,48.63
BILL_AMT1,5.22,8.05,5.88,9.58,3.99,6.54
PAY_3,1.17,3.10,0.77,0.51,0.47,1.21
BILL_AMT2,0.19,3.89,0.47,0.25,0.38,1.04
BILL_AMT3,0.20,2.18,0.04,0.07,0.15,0.53
PAY_5,0.65,0.96,0.17,0.18,0.22,0.44
PAY_AMT1,0.31,0.44,0.25,0.65,0.35,0.40
PAY_4,0.18,0.61,0.30,0.32,0.25,0.33
PAY_AMT2,0.36,0.02,0.09,0.77,0.13,0.27
LIMIT_BAL,0.09,0.15,0.16,0.44,0.40,0.25


**Par mois** (somme des variables du mois)

,Régression logistique,Arbre de décision,RandomForest,MLP,CatBoost,Moyenne des modèles
septembre (M),5.53,8.49,6.14,10.24,4.34,6.95
août (M-1),50.30,56.13,54.49,42.09,46.67,49.94
juillet (M-2),1.48,5.43,0.89,0.81,0.72,1.86
juin (M-3),0.42,0.74,0.44,0.59,0.40,0.52
mai (M-4),0.84,1.74,0.30,0.32,0.30,0.70
avril (M-5),0.64,0.32,0.21,0.50,0.39,0.41
plafond,0.09,0.15,0.16,0.44,0.40,0.25


**Comment lire** : une ligne par variable (ou par mois), une colonne par modèle ; la valeur est la baisse du score, en points, quand la variable est mélangée. Proche de 0 : le modèle ne s'en sert pas ; négative : le mélange ne fait rien perdre (bruit). Si les mois de juin, mai et avril sont proches de 0 pour tous les modèles, les colonnes anciennes n'apportent rien à la codification de septembre.

## 4. Les quatre groupes, parmi les clients sains en août

Parmi les clients du train sains en août (`PAY_2` à -2, -1 ou 0) :
- **nouveaux retards détectés** : en retard en septembre, et mis en retard par **au moins un** modèle ;
- **nouveaux retards manqués** : en retard en septembre, mis en retard par **aucun** modèle ;
- **restés sains** : sains en septembre, mis en retard par aucun modèle (le groupe de comparaison) ;
- **fausses alertes** : sains en septembre, mis en retard par au moins un modèle.

Le second tableau donne, pour chaque modèle, combien de nouveaux retards il détecte, et combien il en est le seul à détecter.

In [8]:
familles_predites = {nom: famille_pay(p) for nom, p in predictions_hors_pli.items()}
alerte = np.column_stack([f == 2 for f in familles_predites.values()])   # mis en retard, modèle par modèle
au_moins_un = alerte.any(axis=1)

famille_aout, famille_sept = famille_pay(X_train["PAY_2"]), famille_pay(y_train)
sain_aout = famille_aout != 2
retard_sept = famille_sept == 2
GROUPES = {
    "Nouveaux retards détectés": sain_aout & retard_sept & au_moins_un,
    "Nouveaux retards manqués": sain_aout & retard_sept & ~au_moins_un,
    "Restés sains": sain_aout & ~retard_sept & ~au_moins_un,
    "Fausses alertes": sain_aout & ~retard_sept & au_moins_un,
}
display(pd.DataFrame({"Clients": {g: int(m.sum()) for g, m in GROUPES.items()}}))

nouveaux = sain_aout & retard_sept
lignes = []
for j, nom in enumerate(familles_predites):
    seul = alerte[:, j] & (alerte.sum(axis=1) == 1)
    lignes.append({"Modèle": nom, "Nouveaux retards détectés": int((alerte[:, j] & nouveaux).sum()),
                   "Dont détectés par lui seul": int((seul & nouveaux).sum()),
                   "Fausses alertes (sains en septembre)": int((alerte[:, j] & sain_aout & ~retard_sept).sum())})
display(pd.DataFrame(lignes).set_index("Modèle"))

,Clients
Nouveaux retards détectés,118
Nouveaux retards manqués,681
Restés sains,17680
Fausses alertes,499


,Nouveaux retards détectés,Dont détectés par lui seul,Fausses alertes (sains en septembre)
Modèle,,,
Régression logistique,80,11,291
Arbre de décision,82,15,332
RandomForest,28,0,48
MLP,0,0,3
CatBoost,87,13,251


**Comment lire** : le premier tableau donne la taille des quatre groupes ; le second, pour chaque modèle, les nouveaux retards qu'il détecte (dont ceux qu'aucun autre ne détecte) et les fausses alertes qu'il donne parmi les clients sains en août. Un modèle qui détecte beaucoup de nouveaux retards au prix de beaucoup de fausses alertes ne sait pas les distinguer : il élargit simplement ses alertes.

## 5. Profil de chaque groupe

Indicateurs calculés sur les données d'origine, pour l'analyse seulement (ils n'entrent dans aucun modèle). Les ratios reprennent les définitions de `docs/colonnes_creees.md` : `ratio_PAY_BILLn` = `PAY_AMTn` / `BILL_AMT(n+1)` × 100, écrêté entre 0 et 200, 100 si aucune facture n'était exigible ; `ratio_BILL_LIMITn` = `BILL_AMTn` / `LIMIT_BAL` × 100, écrêté entre 0 et 200. Convention temporelle : `PAY_AMT1` est le paiement de septembre, qui règle la facture exigible `BILL_AMT2` (solde de fin août) ; `PAY_AMT2`, celui d'août, règle `BILL_AMT3`.

- **paiement de septembre** : `ratio_PAY_BILL1`, part de la facture exigible payée ; et part des clients qui ne paient **rien** alors qu'une facture était due ;
- **paiement d'août** : `ratio_PAY_BILL2`, pour le décalage d'un mois entre paiement et codification ;
- **facture exigible de septembre** (`BILL_AMT2`) et **plafond** (`LIMIT_BAL`), en médiane ;
- **utilisation du plafond** fin septembre (`ratio_BILL_LIMIT1`) et part des clients **au-dessus de leur plafond** ;
- **retard passé** : part des clients qui ont eu au moins une codification de 2 et plus d'avril à juillet (`PAY_3` à `PAY_6`) ;
- **codification d'août** : répartition entre -2, -1 et 0.

In [9]:
o = df.loc[X_train.index]   # données d'origine des clients du train (codifications détaillées)

def ratio_paiement(n):
    facture = o[f"BILL_AMT{n + 1}"]
    return (o[f"PAY_AMT{n}"] / facture.where(facture > 0) * 100).clip(lower=0, upper=200).fillna(100)

INDICATEURS = pd.DataFrame({
    "Ratio de paiement de septembre (%)": ratio_paiement(1),
    "Rien payé en septembre, facture due (%)": ((o["PAY_AMT1"] == 0) & (o["BILL_AMT2"] > 0)) * 100,
    "Ratio de paiement d'août (%)": ratio_paiement(2),
    "Facture exigible de septembre (NT$)": o["BILL_AMT2"],
    "Plafond (NT$)": o["LIMIT_BAL"],
    "Utilisation du plafond fin septembre (%)": (o["BILL_AMT1"] / o["LIMIT_BAL"] * 100).clip(lower=0, upper=200),
    "Au-dessus du plafond fin septembre (%)": (o["BILL_AMT1"] > o["LIMIT_BAL"]) * 100,
    "Retard passé d'avril à juillet (%)": (o[[f"PAY_{n}" for n in range(3, 7)]] >= 2).any(axis=1) * 100,
}, index=o.index).astype(float)
MEDIANES = ["Ratio de paiement de septembre (%)", "Ratio de paiement d'août (%)", "Facture exigible de septembre (NT$)",
            "Plafond (NT$)", "Utilisation du plafond fin septembre (%)"]

lignes = []
for groupe, masque in GROUPES.items():
    g = INDICATEURS[masque]
    ligne = {"Groupe": groupe, "Clients": int(masque.sum())}
    for col in INDICATEURS.columns:
        ligne[col + (" — médiane" if col in MEDIANES else "")] = round(g[col].median() if col in MEDIANES else g[col].mean(), 1)
    for c in (-2, -1, 0):
        ligne[f"Août à {c} (%)"] = round((o.loc[masque, "PAY_2"] == c).mean() * 100, 1)
    lignes.append(ligne)
display(pd.DataFrame(lignes).set_index("Groupe").T)

Groupe,Nouveaux retards détectés,Nouveaux retards manqués,Restés sains,Fausses alertes
Clients,118.0,681.0,17680.0,499.0
Ratio de paiement de septembre (%) — médiane,6.5,5.4,13.6,7.5
"Rien payé en septembre, facture due (%)",0.0,0.1,0.1,0.0
Ratio de paiement d'août (%) — médiane,9.9,5.7,12.0,9.4
Facture exigible de septembre (NT$) — médiane,27093.0,45880.0,22927.0,29954.0
Plafond (NT$) — médiane,50000.0,80000.0,150000.0,60000.0
Utilisation du plafond fin septembre (%) — médiane,61.4,78.2,27.8,65.4
Au-dessus du plafond fin septembre (%),5.9,12.6,6.1,6.4
Retard passé d'avril à juillet (%),97.5,16.7,10.4,89.4
Août à -2 (%),0.0,0.0,11.6,0.0


**Comment lire** : une colonne par groupe, une ligne par indicateur (médiane pour les ratios et les montants, part en % pour les autres). On compare surtout les **nouveaux retards manqués** aux **nouveaux retards détectés** (qu'est-ce qui les sépare ?) et aux **restés sains** (leur ressemblent-ils ?).

## 6. Les nouveaux retards manqués ressemblent-ils aux clients restés sains ?

Pour chaque indicateur, l'**écart standardisé** (différence des moyennes divisée par l'écart-type commun des deux groupes), comme dans l'analyse des défauts manqués du machine learning de la deuxième itération, entre les nouveaux retards manqués et deux groupes de comparaison : les nouveaux retards détectés, et les clients restés sains. Repères habituels : un écart de moins de 0,2 est négligeable, de 0,2 à 0,5 faible, de 0,5 à 0,8 moyen, au-delà de 0,8 fort.

In [10]:
def ecart_standardise(a, b):
    commun = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return (a.mean() - b.mean()) / commun if commun > 0 else 0.0

manques = INDICATEURS[GROUPES["Nouveaux retards manqués"]]
lignes = []
for col in INDICATEURS.columns:
    lignes.append({"Indicateur": col,
                   "Manqués - détectés": round(ecart_standardise(manques[col], INDICATEURS.loc[GROUPES["Nouveaux retards détectés"], col]), 2),
                   "Manqués - restés sains": round(ecart_standardise(manques[col], INDICATEURS.loc[GROUPES["Restés sains"], col]), 2)})
display(pd.DataFrame(lignes).set_index("Indicateur"))

,Manqués - détectés,Manqués - restés sains
Indicateur,,
Ratio de paiement de septembre (%),0.35,-0.58
"Rien payé en septembre, facture due (%)",0.05,0.01
Ratio de paiement d'août (%),0.33,-0.55
Facture exigible de septembre (NT$),0.34,0.23
Plafond (NT$),0.60,-0.49
Utilisation du plafond fin septembre (%),0.10,0.60
Au-dessus du plafond fin septembre (%),0.23,0.22
Retard passé d'avril à juillet (%),-2.81,0.18


**Comment lire** :
- **si les nouveaux retards manqués sont proches des restés sains** sur la plupart des indicateurs (écarts faibles dans la seconde colonne), rien dans les données ne les annonce : leur retard vient d'une information que le dataset ne montre pas (issue 1) ;
- **s'ils s'en écartent nettement** sur un indicateur (par exemple un paiement de septembre ou d'août bien plus faible), ce signe est dans les données, et les modèles ne l'exploitent pas : piste pour les aider (issue 2) ;
- la première colonne dit ce qui sépare les retards manqués de ceux que les modèles détectent.

## 7. Conclusion

*À écrire après lecture des résultats : issue 1 ou 2, et ce que cela dit de la codification des nouveaux retards.*